In [1]:
%load_ext autoreload
%autoreload 2

# Knowledge graph generation

Builds the math knowledge graph from the raw Question/Answer/Tag skeleton, in three stages:

1. **Enrich** — one LLM call per question adds `Concept` nodes and stores the solution's
   resolution steps on the `Answer` node.
2. **Resolve concepts** — one global pass clusters concept name variants into canonical nodes.
   This is the step that makes the `Concept` layer actually connect problems; per-question
   extraction alone leaves ~95% of concepts attached to a single question.
3. **Render** — PyVis HTML.

The graph has three node layers on purpose: `Tag` (macro topic) → `Question`/`Answer`
(substance) → `Concept` (shared mid-layer). See `docs/usage.md`.

In [9]:
import json
import os
from collections import Counter, defaultdict
from datetime import datetime

from dotenv import load_dotenv

from math_assistant_agent.data import load_graph_json, save_graph_json
from math_assistant_agent.enrichment import (
    apply_concept_resolution,
    build_groq_client,
    enrich_graph_records,
    extract_graph_entities_groq,
    resolve_concepts,
    resolve_concepts_groq,
)
from math_assistant_agent.visualization import render_graph

load_dotenv()
GROQ_API_KEY = os.environ.get("GROQ_API_KEY")

DATA_DIR = "/home/cdrleo/Desktop/ai-projects/math-assistant-agent/data"
INGESTION_DATE = datetime.now().strftime("%Y-%m-%d-%H")

# Raw Question/Answer/Tag skeleton. Never written to, so a run can always start over from it.
RAW_GRAPH_PATH = f"{DATA_DIR}/graph_math_2026-07-18-15.json"

# Enriched output. enrich_graph_records checkpoints here as it goes.
KB_GRAPH_PATH = f"{DATA_DIR}/graph_math_kb_{INGESTION_DATE}.json"

print(f"raw:     {RAW_GRAPH_PATH}")
print(f"output:  {KB_GRAPH_PATH}")

raw:     /home/cdrleo/Desktop/ai-projects/math-assistant-agent/data/graph_math_2026-07-18-15.json
output:  /home/cdrleo/Desktop/ai-projects/math-assistant-agent/data/graph_math_kb_2026-07-18-15.json


In [10]:
def graph_report(graph_data, title="graph"):
    """Print the health metrics that define whether the graph is navigable.

    Leaf ratio and concept sharing are the ones to watch: a graph full of degree-1
    nodes looks dense when rendered but has nothing to traverse.
    """
    nodes, edges = graph_data["nodes"], graph_data["edges"]

    adjacency = defaultdict(set)
    for edge in edges:
        adjacency[edge["source"]].add(edge["target"])
        adjacency[edge["target"]].add(edge["source"])

    labels = {n["id"]: n["label"] for n in nodes}
    names = {n["id"]: n["properties"].get("name", n["id"]) for n in nodes}

    leaves = sum(1 for n in nodes if len(adjacency[n["id"]]) <= 1)
    concepts = [n for n in nodes if n["label"] == "Concept"]
    lonely = [c for c in concepts if len(adjacency[c["id"]]) <= 1]
    steps = sum(len(n["properties"].get("resolution_steps", [])) for n in nodes)

    print(f"=== {title}: {len(nodes)} nodes / {len(edges)} edges")
    print(f"    labels     {dict(Counter(n['label'] for n in nodes))}")
    print(f"    edges      {dict(Counter(e['type'] for e in edges))}")
    print(f"    leaves     {leaves}/{len(nodes)} ({leaves / len(nodes):.0%})")
    if concepts:
        shared = len(concepts) - len(lonely)
        print(f"    concepts   {shared}/{len(concepts)} touch >1 question ({shared / len(concepts):.0%})")
    print(f"    steps      {steps} stored on Answer nodes")

    dangling = [e for e in edges if e["source"] not in labels or e["target"] not in labels]
    if dangling:
        print(f"    ⚠️  {len(dangling)} dangling edges")

    top = sorted(((len(v), k) for k, v in adjacency.items()), reverse=True)[:8]
    print("    top hubs   " + ", ".join(f"{names[k]}({labels[k][:4]}:{d})" for d, k in top))


graph_report(load_graph_json(RAW_GRAPH_PATH), "raw skeleton")

=== raw skeleton: 37 nodes / 30 edges
    labels     {'Question': 10, 'Answer': 10, 'Tag': 17}
    edges      {'HAS_ACCEPTED_ANSWER': 10, 'TAGGED_WITH': 20}
    leaves     25/37 (68%)
    steps      0 stored on Answer nodes
    top hubs   question_106560(Ques:5), question_22927(Ques:4), question_182412(Ques:4), question_60375(Ques:3), question_34059(Ques:3), question_20740(Ques:3), question_20493(Ques:3), pr.probability(Tag:2)


## 1. Enrich

One LLM call per question. Adds `Concept` nodes (each with a description used later for
resolution) and writes the solution's `resolution_steps` onto the `Answer` node.

Safe to re-run: questions whose `Answer` already carries `resolution_steps` are skipped, so
after a rate-limit crash just execute the cell again to resume. Progress is checkpointed to
`KB_GRAPH_PATH` every 10 questions and immediately before any exception propagates.

In [11]:
client = build_groq_client(api_key=GROQ_API_KEY)

In [12]:
# Reads RAW_GRAPH_PATH, writes only to KB_GRAPH_PATH — the raw skeleton stays pristine.
# To resume an interrupted run, pass KB_GRAPH_PATH as the first argument instead.
graph_data = enrich_graph_records(
    RAW_GRAPH_PATH,
    client=client,
    extract_fn=extract_graph_entities_groq,
    sleep_seconds=10,
    checkpoint_path=KB_GRAPH_PATH,
)

graph_report(graph_data, "after enrichment")

Extracted concepts: ['Logarithmic derivative of a polynomial', 'Residue theorem', 'Argument principle', 'Unit circle as a typical contour for random polynomials', 'Pizza slice contour for angular distribution']
Extracted concepts: ['Neukirch-Uchida theorem', 'Etale fundamental group', 'Section conjecture', 'Anabelian geometry', 'ABC conjecture']
Extracted concepts: ['Expositiones Mathematicae', 'Expository paper', 'T. Bühler paper']
Extracted concepts: ['Axiom of Choice', 'Surjection Splitting', 'Internal Category', 'Lie Algebra Object', 'Poincaré–Birkhoff–Witt Theorem']
Extracted concepts: ['Infinitely differentiable function', 'Derivative', 'Polynomial function', 'Baire category theorem', 'Closed set']
Extracted concepts: ['Relative Kunneth Formula', 'Relative Homology', 'Path Components', 'Product of Spaces', 'Homology Groups']
Extracted concepts: ['Enhanced measurable space', 'Localizable space', 'Compact strictly localizable space', 'Measurable locale', 'Commutative von Neumann al

RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-20b` in organization `org_01kxsa2evmf07948nc17h47ss2` service tier `on_demand` on tokens per day (TPD): Limit 200000, Used 197452, Requested 3728. Please try again in 8m29.76s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

In [16]:
with open(KB_GRAPH_PATH, "r") as f:
    graph_data = json.load(f)

# graph_data

## 2. Resolve concepts

Extraction runs one question at a time, so the same idea comes back phrased differently each
time ("Kunneth Formula" / "Künneth theorem"). This pass clusters every concept at once, using
each one's description as disambiguation context, and is what turns `Concept` from a layer of
dead ends into one that links problems.

Cost is roughly `n_concepts / 80 + 1` calls, not one per question.

The two steps are deliberately separate — inspect (and if needed hand-edit) `alias_map` before
applying it. `apply_concept_resolution` makes no LLM calls, so it can be re-run for free.

In [17]:
alias_map = resolve_concepts(
    graph_data,
    client=client,
    resolve_fn=resolve_concepts_groq,
)

# Review what it wants to merge before touching the graph. Watch for over-merging: a specific
# result folded into a broad area (e.g. "Cayley-Hamilton Theorem" -> "Linear Algebra") is a
# mistake — delete that entry from alias_map rather than accepting it.
merges = defaultdict(list)
for alias, canonical in alias_map.items():
    if alias != canonical:
        merges[canonical].append(alias)

print(f"{sum(len(v) for v in merges.values())} names merged into {len(merges)} concepts\n")
for canonical, aliases in sorted(merges.items()):
    print(f"  {canonical}")
    for alias in sorted(aliases):
        print(f"      <- {alias}")

Pass 1: resolving 1-36 of 36...


RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-20b` in organization `org_01kxsa2evmf07948nc17h47ss2` service tier `on_demand` on tokens per day (TPD): Limit 200000, Used 197097, Requested 3803. Please try again in 6m28.799999999s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

In [ ]:
apply_concept_resolution(graph_data, alias_map)
save_graph_json(graph_data, path=KB_GRAPH_PATH)

graph_report(graph_data, "after concept resolution")

## 3. Render

Nodes are colored by layer: `Tag` orange (macro topic), `Question` red / `Answer` green
(substance), `Concept` purple (shared mid-layer). Resolution steps and merged concept aliases
are node properties now, so they show on hover rather than as their own nodes.

In [18]:
# Re-read from disk so what gets rendered is exactly what was persisted.
graph_data = load_graph_json(KB_GRAPH_PATH)

render_graph(graph_data, output_path=f"{DATA_DIR}/graph_math_kb_{INGESTION_DATE}.html")

Rendering visualization for 73 nodes and 66 edges...
✅ Graph rendered successfully! Open '/home/cdrleo/Desktop/ai-projects/math-assistant-agent/data/graph_math_kb_2026-07-18-15.html' in your browser.


## Spot check — is it actually navigable?

The point of the `Concept` layer is multi-hop reasoning: reaching a related problem that shares
no tag with the one you started from. If the concepts below each connect only one question, the
resolution pass didn't do its job — re-check the `alias_map` output above.

In [ ]:
titles = {
    n["id"]: n["properties"]["title"] for n in graph_data["nodes"] if n["label"] == "Question"
}
concept_names = {
    n["id"]: n["properties"]["name"] for n in graph_data["nodes"] if n["label"] == "Concept"
}

concept_to_questions = defaultdict(list)
for edge in graph_data["edges"]:
    if edge["type"] == "APPLIES_TO_PROBLEM":
        concept_to_questions[edge["source"]].append(edge["target"])

# The concepts that bridge the most problems — the graph's actual connective tissue.
bridges = sorted(concept_to_questions.items(), key=lambda kv: -len(kv[1]))

print("Concepts linking the most questions:\n")
for concept_id, question_ids in bridges[:10]:
    if len(question_ids) < 2:
        break
    print(f"  {concept_names[concept_id]}  ({len(question_ids)} questions)")
    for question_id in question_ids[:4]:
        print(f"      - {titles[question_id][:80]}")
    print()

In [ ]:
# Sanity check one enriched answer end to end: steps now live on the Answer node.
answers = [
    n
    for n in graph_data["nodes"]
    if n["label"] == "Answer" and n["properties"].get("resolution_steps")
]
print(f"{len(answers)} answers carry resolution steps\n")

sample = answers[0]
for step in sample["properties"]["resolution_steps"]:
    print(f"  {step['step_number']}. {step['description']}")
    if step["formula_latex"]:
        print(f"     $ {step['formula_latex']} $")